# Simulation I: Homoscedastic Errors
**Zhong and Wang (2024), pp. 608-609, Tables 1-3**  
Created 04092026; notebook and paper audit 08092026.

This notebook is the sole Python simulation source, like the root `demo.ipynb`.
It directly reuses the original root `dqAux.py`; no separate simulation Python
script is required. The R file supplies the residual density for DPLQR inference
and the optional PLAQR comparison stage.

**Execution order:** select the repository `.venv` kernel. Run Sections **1-7**
first (in VS Code, use **Run All Above** on the Section 8 code cell). Section 8
runs LQR and DPLQR and writes CSV results; Section 9 optionally adds PLAQR.
After restarting a kernel, run the setup cells again. **Restart Kernel and Run
All** runs the whole configured experiment, including PLAQR.

The active profile is a **first pass**: two repetitions, all three cases,
n=1000, 1,000 test observations, the median quantile, and a fixed network trained
for up to 100 epochs. Conflicting full-replication assignments are preserved in
Section 2 with the searchable comment marker **`# *%%* FULL REPLICATION:`**.
Outputs go to `first-pass-run/`. No numerical results are prefilled.
See [README.md](README.md) for the remaining paper/source ambiguities.

## 1. Imports and repository paths

In [1]:
"""Reconstruct Zhong and Wang (2024), Section 5.1, Tables 1-3.

This notebook adapts the authors' demo.ipynb workflow and imports the original
DPLQR components from the repository's dqAux.py. The simulation DGP, repeated
sampling, two-coefficient inference, and table construction are added here.
"""

from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
from types import CodeType, FunctionType, SimpleNamespace
from IPython.display import display, Markdown

import numpy as np
import pandas as pd
from scipy.stats import norm, t
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import torch
from torch import nn
from torchtuples import Model
import torchtuples as tt

# Find the original repository from the notebook folder or repository root.
working_directory = Path.cwd().resolve()
ROOT = next((p for p in (working_directory, *working_directory.parents)
             if (p / "dqAux.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find dqAux.py. Open this notebook inside the dplqr repository "
        "and restart the kernel with the repository .venv interpreter.")
SCRIPT_DIR = ROOT / "results" / "2026-09-04-homoscedastic-simulation"
sys.path.insert(0, str(ROOT))
from dqAux import checkErrorMean, checkLoss, covNet, dqNetSparse

THETA = np.array([1.0, -1.0])
CASES = (1, 2, 3)
TAUS = (0.25, 0.50, 0.75)
SAMPLE_SIZES = (1000, 2000)
METHODS = ("LQR", "DPLQR")
PIPELINE_VERSION = "2026-09-08-notebook"
Z975 = norm.ppf(0.975)

R is also used for the residual density in DPLQR inference, as specified on p. 608. Set `RSCRIPT` in the environment if it is not detected automatically.

In [2]:
from __future__ import annotations

def find_rscript():
    executable = os.environ.get("RSCRIPT") or shutil.which("Rscript")
    if executable:
        return str(executable)
    candidates = sorted(Path(os.environ.get("ProgramFiles", "C:/Program Files")).glob("R/R-*/bin/Rscript.exe"))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError("Rscript is required for the paper's stats::density estimator. Set RSCRIPT to its path.")


def residual_density_zero(residual):
    # Paper p.608 specifies R stats::density, rather than scipy gaussian_kde.
    result = subprocess.run([find_rscript(), "--vanilla", str(SCRIPT_DIR / "simulate_homoscedastic.R"),
                             "--density-only"], input="\n".join(format(v, ".17g") for v in residual),
                            capture_output=True, text=True, check=True)
    density = float(result.stdout.strip())
    if not np.isfinite(density) or density <= 0:
        raise ValueError("Residual density at zero must be finite and positive")
    return density

## 2. Experiment settings - FIRST PASS
The active settings use two repetitions, n=1000, all three cases, 1,000 test
observations, and tau=0.5. Fixed tuning uses depth=2, width=32, 100 epochs,
batch size=128, patience=15, learning rate=0.005, and one CPU thread.

Search for **`# *%%* FULL REPLICATION:`** to find the commented original values.
For restoration, uncomment each original assignment and comment its first-pass
replacement. In grid mode, the six network tuning parameters are searched over
the supplement's full candidate set; their individual fixed values are ignored.
The R stage in Section 9 receives the current settings automatically.

In [3]:
if "SimpleNamespace" not in globals() or "find_rscript" not in globals():
    raise RuntimeError("Run both code cells in Section 1 before the experiment settings.")

# ACTIVE PROFILE: FIRST PASS
# *%%* FULL REPLICATION marks preserved original assignments.
# To restore one, remove its comment prefix and comment out the active line below.
# Grid mode chooses epochs/patience/learning rate from Table 13; their preserved
# individual values below are the original fixed-mode defaults.
args = SimpleNamespace(
    # *%%* FULL REPLICATION: repetitions=200,
    repetitions=2,
    # *%%* FULL REPLICATION: sample_sizes=[1000, 2000],
    sample_sizes=[1000],
    # *%%* FULL REPLICATION: test_size=5000,
    test_size=1000,
    cases=[1, 2, 3],
    # *%%* FULL REPLICATION: taus=[0.25, 0.50, 0.75],
    taus=[0.5],
    seed=20260904,
    # *%%* FULL REPLICATION: hyperparameter_mode="grid",
    hyperparameter_mode="fixed",
    depth=2,
    width=32,
    # *%%* FULL REPLICATION: epochs=500,
    epochs=100,
    batch_size=128,
    # *%%* FULL REPLICATION: patience=200,
    patience=15,
    # *%%* FULL REPLICATION: learning_rate=0.001,
    learning_rate=0.005,
    threads=1,
    fail_fast=True,
    restart=False,
    export_r_data=True,
    # *%%* FULL REPLICATION: output_dir=SCRIPT_DIR / "full-run",
    output_dir=SCRIPT_DIR / "first-pass-run",
)

print("ACTIVE PROFILE: FIRST PASS - fixed tuning, two repetitions, median quantile")
print("Python:", sys.executable)
print("Rscript:", find_rscript())
print("Settings:", vars(args))

ACTIVE PROFILE: FIRST PASS - fixed tuning, two repetitions, median quantile
Python: C:\Users\Tiansui Tu\Documents\GitHub\dplqr\.venv\Scripts\python.exe
Rscript: C:\Program Files\R\R-4.6.1\bin\Rscript.exe
Settings: {'repetitions': 2, 'sample_sizes': [1000], 'test_size': 1000, 'cases': [1, 2, 3], 'taus': [0.5], 'seed': 20260904, 'hyperparameter_mode': 'fixed', 'depth': 2, 'width': 32, 'epochs': 100, 'batch_size': 128, 'patience': 15, 'learning_rate': 0.005, 'threads': 1, 'fail_fast': True, 'restart': False, 'export_r_data': True, 'output_dir': WindowsPath('C:/Users/Tiansui Tu/Documents/GitHub/dplqr/results/2026-09-04-homoscedastic-simulation/first-pass-run')}


## 3. Data-generating process
The nonlinear term in Case 2 includes **sqrt(z6 + 0.5)**. The true conditional quantile is X theta + m(Z) + the t(3) quantile.

In [4]:
from __future__ import annotations

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)


def nonlinear_truth(z: np.ndarray, case: int) -> np.ndarray:
    if case == 1:
        return 0.56 * z.sum(axis=1)
    if case == 2:
        inside = ((z[:, 0] - 1) ** 2 - z[:, 1] ** 2
                  + 3 * np.abs(z[:, 2] - 1) + 0.6 * np.sin(np.pi * z[:, 3])
                  + np.log(z[:, 4] + 0.5) + np.sqrt(z[:, 5] + 0.5)
                  + 3 * np.cos(0.1 * np.pi * z[:, 6])
                  + 3 * (z[:, 7] - 1 + np.abs(z[:, 7] - 1)))
        return 0.82 * inside
    if case == 3:
        inside = (np.exp(z[:, 0] * (1 + z[:, 1] - np.pi * z[:, 2] * z[:, 3]) / 2)
                  * (z[:, 4] + 0.2)
                  + z[:, 4] * (z[:, 3] - 0.3) / (np.abs(2 * z[:, 3] - 1) + 1)
                  + 2 * np.sin(z[:, 4]) * np.abs(z[:, 4] * z[:, 5] - 0.6)
                  + np.log(z[:, 5] + z[:, 6] * z[:, 7]))
        return 0.61 * inside
    raise ValueError(f"Unknown case: {case}")


def generate_covariates(n: int, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    # Gaussian copula with equicorrelation 0.5 and Uniform[0, 2] margins.
    covariance = np.full((10, 10), 0.5)
    np.fill_diagonal(covariance, 1.0)
    z_tilde = 2 * norm.cdf(rng.multivariate_normal(np.zeros(10), covariance, size=n))
    z = z_tilde[:, :8]
    x = np.column_stack((z_tilde[:, 8] > 1, z_tilde[:, 9])).astype(float)
    return x, z


def generate_dataset(n: int, case: int, rng: np.random.Generator, include_error: bool = True):
    x, z = generate_covariates(n, rng)
    m = nonlinear_truth(z, case)
    error = rng.standard_t(df=3, size=n) if include_error else np.zeros(n)
    y = x @ THETA + m + error
    return x, z, y, m


def export_dataset(path: Path, x, z, y, m):
    frame = pd.DataFrame(np.column_stack((y, x, z, m)),
                         columns=["y", "x1", "x2", *[f"z{i}" for i in range(1, 9)], "true_m"])
    frame.to_csv(path, index=False, compression="gzip")


def as_numpy(value):
    return value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

## 4. Linear quantile regression (LQR)
The formula interface follows `demo.ipynb`; Simulation I has two linear covariates and eight nonlinear covariates.

In [5]:
from __future__ import annotations

def fit_lqr(x_train, z_train, y_train, x_test, z_test, tau):
    columns = ["x1", "x2", *[f"z{i}" for i in range(1, 9)]]
    train = pd.DataFrame(np.column_stack((x_train, z_train)), columns=columns)
    train.insert(0, "y", y_train)
    test = pd.DataFrame(np.column_stack((x_test, z_test)), columns=columns)
    fit = smf.quantreg("y ~ x1 + x2 + z1 + z2 + z3 + z4 + z5 + z6 + z7 + z8", train).fit(
        q=tau, max_iter=5000)
    prediction = np.asarray(fit.predict(test))
    theta = fit.params.loc[["x1", "x2"]].to_numpy()
    m_hat = prediction - x_test @ theta
    return theta, fit.bse.loc[["x1", "x2"]].to_numpy(), prediction, m_hat

## 5. DPLQR training and validation
The original `dqNetSparse` and `checkLoss` are reused. The paper specifies PyTorch initialization and grid selection. Neural weights and biases are clipped after training. The original depth and random sparsity conventions are retained and explained in the README.

In [6]:
from __future__ import annotations

import torchtuples as tt  # Needed here when loading this definition cell separately.

class EarlyStopInMemory(tt.callbacks.Callback):
    def __init__(self, patience):
        self.patience = patience

    def on_fit_start(self):
        self.best = math.inf
        self.best_state = None
        self.best_epoch = 0
        self.since_best = 0
        self.epochs_run = 0

    def on_epoch_end(self):
        self.epochs_run += 1
        score = self.model.val_metrics.scores["loss"]["score"][-1]
        if not np.isfinite(score):
            raise FloatingPointError("Non-finite validation loss")
        if score < self.best:
            self.best = float(score)
            self.best_state = {k: v.detach().clone() for k, v in self.model.net.state_dict().items()}
            self.best_epoch = self.epochs_run
            self.since_best = 0
        else:
            self.since_best += 1
        return self.since_best >= self.patience

    def on_fit_end(self):
        self.model.net.load_state_dict(self.best_state)


def clip_neural_weights(net):
    # Pages 605/608: W_k includes weights and biases; leave theta and masks alone.
    with torch.no_grad():
        for name, parameter in net.named_parameters():
            if "nonpar" in name and name.rsplit(".", 1)[-1] in ("weight", "bias"):
                parameter.clamp_(-1.0, 1.0)


def tensor_pair(x, z):
    return torch.tensor(x, dtype=torch.float32), torch.tensor(z, dtype=torch.float32)


def train_dplqr_once(x_train, z_train, y_train, x_val, z_val, y_val, tau, hp, seed):
    seed_all(seed)
    xs = StandardScaler().fit(z_train)
    ztr, zv = xs.transform(z_train), xs.transform(z_val)
    train_x, val_x = tensor_pair(x_train, ztr), tensor_pair(x_val, zv)
    net = dqNetSparse(2, 8, torch.zeros((1, 2), dtype=torch.float32),
                      [hp["depth"], hp["width"]], sparseRatio=0.5)
    # Paper p.607 uses PyTorch initialization; the concrete demo uses LQR instead.
    net.linLinear.reset_parameters()
    model = Model(net, checkLoss(tau=tau), device="cpu")
    model.optimizer.set_lr(hp["lr"])
    stopper = EarlyStopInMemory(hp["patience"])
    model.fit(train_x, torch.tensor(y_train[:, None], dtype=torch.float32), hp["batch_size"],
              hp["epochs"], [stopper], False,
              val_data=(val_x, torch.tensor(y_val[:, None], dtype=torch.float32)),
              val_batch_size=hp["batch_size"])
    return model, xs, stopper


def select_dplqr(x_train, z_train, y_train, x_val, z_val, y_val, tau, config, seed):
    profiles = [config]
    if config["mode"] == "grid":
        profiles = [dict(mode="grid", depth=d, width=w, epochs=e, batch_size=b, patience=p, lr=lr)
                    for d, w, e, b, p, lr in itertools.product(
                        [1, 2, 3, 5], [16, 32, 64], [200, 500], [64, 128], [200, 300],
                        [0.001, 0.005, 0.01, 0.05])]
    best = None
    for candidate_index, hp in enumerate(profiles, start=1):
        model, scaler, stopper = train_dplqr_once(
            x_train, z_train, y_train, x_val, z_val, y_val, tau, hp, seed)
        clip_neural_weights(model.net)
        validation = as_numpy(model.predict(tensor_pair(x_val, scaler.transform(z_val)))).reshape(-1)
        score = checkErrorMean(validation.reshape(-1, 1), y_val.reshape(-1, 1), tau=tau)
        if best is None or score < best[0]:
            best = score, model, scaler, stopper, hp
        if len(profiles) > 1:
            print(f"  grid candidate {candidate_index}/{len(profiles)}; validation loss={score:.6g}", flush=True)
    return best[1:]


def fit_projection(z_train, target_train, z_val, target_val, hp, seed, binary):
    seed_all(seed)
    # Section 4 projects both linear covariates by squared error.
    loss = nn.MSELoss()
    net = covNet(8, [hp["depth"], hp["width"]], logic=binary)
    model = Model(net, loss, device="cpu")
    model.optimizer.set_lr(hp["lr"])
    stopper = EarlyStopInMemory(hp["patience"])
    model.fit(torch.tensor(z_train, dtype=torch.float32), torch.tensor(target_train[:, None], dtype=torch.float32),
              hp["batch_size"], hp["epochs"], [stopper], False,
              val_data=(torch.tensor(z_val, dtype=torch.float32),
                        torch.tensor(target_val[:, None], dtype=torch.float32)),
              val_batch_size=hp["batch_size"])
    return model, stopper

## 6. DPLQR inference
The two auxiliary `covNet` projections use squared error. Equation (14) combines the residual density at zero and the covariance of projected linear covariates. The estimator uses 80% of observations, so this is also the sample used to scale its covariance.

In [7]:
from __future__ import annotations

def dplqr_standard_errors(model, scaler, hp, x_train, z_train, y_train, x_val, z_val, y_val, tau, seed):
    # Section 5.1 uses 80% for estimation; the covariance scales by that size.
    x_all, z_all, y_all = x_train, scaler.transform(z_train), y_train
    pred = as_numpy(model.predict(tensor_pair(x_all, z_all))).reshape(-1)
    residual = y_all - pred
    projected, epochs = [], []
    for column, binary in [(0, True), (1, False)]:
        projection, stopper = fit_projection(
            scaler.transform(z_train), x_train[:, column], scaler.transform(z_val), x_val[:, column],
            hp, seed + column, binary)
        projected.append(as_numpy(projection.predict(torch.tensor(z_all, dtype=torch.float32))).reshape(-1))
        epochs.append(stopper.epochs_run)
    v = x_all - np.column_stack(projected)
    omega = np.cov(v, rowvar=False, ddof=1)
    f_zero = residual_density_zero(residual)
    covariance = tau * (1 - tau) * np.linalg.inv(omega) / (f_zero**2 * len(y_all))
    return np.sqrt(np.diag(covariance)), f_zero, epochs


def fit_dplqr(x_train, z_train, y_train, x_val, z_val, y_val, x_test, z_test, tau, config, seed):
    model, scaler, stopper, hp = select_dplqr(
        x_train, z_train, y_train, x_val, z_val, y_val, tau, config, seed)
    clip_neural_weights(model.net)
    test_x = tensor_pair(x_test, scaler.transform(z_test))
    prediction = as_numpy(model.predict(test_x)).reshape(-1)
    theta = model.net.linLinear.weight.detach().numpy().reshape(-1)
    m_hat = prediction - x_test @ theta
    se, density, projection_epochs = dplqr_standard_errors(
        model, scaler, hp, x_train, z_train, y_train, x_val, z_val, y_val, tau, seed + 500_000)
    return theta, se, prediction, m_hat, hp, stopper.epochs_run, density, projection_epochs

## 7. Tables and Monte Carlo loop
Each completed fit is checkpointed. Resume checks use the live notebook
function definitions and scientific constants, plus the original helper/R
sources. Saved outputs and execution counts do not affect the code identity.
Table 3 uses relative MSE without taking a square root.

In [8]:
from __future__ import annotations

def summarize(raw: pd.DataFrame, output: Path):
    summary = raw.groupby(["case", "n", "tau", "method"], as_index=False).agg(
        bias=("theta1", lambda x: x.mean() - THETA[0]),
        sd=("theta1", "std"), coverage=("covered_theta1", "mean"), rmse=("rmse_m", "mean"),
        successful_reps=("rep", "count"))
    summary.to_csv(output / "simulation_summary_long.csv", index=False)

    display = summary.copy()
    display["bias_sd"] = [f"{b:.4f} ({s:.4f})" if np.isfinite(s) else f"{b:.4f} (NA)"
                          for b, s in zip(display.bias, display.sd)]

    index = ["case", "n"]
    ordered = pd.MultiIndex.from_product([sorted(raw.tau.unique()), METHODS])
    for name, value in [("table1_bias", "bias"), ("table1_sd", "sd"),
                        ("table2_coverage", "coverage"), ("table3_rmse", "rmse")]:
        table = summary.pivot(index=index, columns=["tau", "method"], values=value).reindex(columns=ordered)
        table.columns = [f"tau_{tau:.2f}_{method}" for tau, method in table.columns]
        table.reset_index().to_csv(output / f"{name}.csv", index=False)
    table1 = display.pivot(index=index, columns=["tau", "method"], values="bias_sd").reindex(columns=ordered)
    table1.columns = [f"tau_{tau:.2f}_{method}" for tau, method in table1.columns]
    table1.reset_index().to_csv(output / "table1_bias_sd.csv", index=False)
    return summary


def validate_args(args):
    for name in ("repetitions", "test_size", "depth", "width", "epochs", "batch_size", "patience", "threads"):
        if getattr(args, name) < 1:
            raise ValueError(f"{name} must be positive")
    if args.learning_rate <= 0 or args.seed < 0:
        raise ValueError("learning_rate must be positive and seed nonnegative")
    for name in ("cases", "sample_sizes", "taus"):
        values = getattr(args, name)
        if not values or len(set(values)) != len(values):
            raise ValueError(f"{name} must be nonempty and contain no duplicates")
    if any(n < 50 for n in args.sample_sizes) or any(c not in CASES for c in args.cases):
        raise ValueError("Use sample sizes >= 50 and cases 1, 2, 3")
    if any(not 0 < tau < 1 for tau in args.taus):
        raise ValueError("Quantiles must be strictly between zero and one")


def _implementation_digest():
    """Hash live definitions, ignoring notebook cell filenames and line numbers."""
    def canonical(value):
        if isinstance(value, CodeType):
            return {"bytecode": value.co_code.hex(),
                    "constants": [canonical(v) for v in value.co_consts],
                    "names": value.co_names, "variables": value.co_varnames,
                    "free_variables": value.co_freevars, "cell_variables": value.co_cellvars,
                    "argcount": value.co_argcount, "posonlyargcount": value.co_posonlyargcount,
                    "kwonlyargcount": value.co_kwonlyargcount, "flags": value.co_flags,
                    "exception_table": getattr(value, "co_exceptiontable", b"").hex()}
        if isinstance(value, (set, frozenset)):
            return {"set": sorted((canonical(v) for v in value), key=lambda v: json.dumps(v, sort_keys=True))}
        if isinstance(value, (tuple, list)):
            return [canonical(v) for v in value]
        if isinstance(value, dict):
            return {str(k): canonical(v) for k, v in sorted(value.items())}
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        raise TypeError(f"Cannot fingerprint code constant of type {type(value).__name__}")

    def function_signature(function):
        return {"code": canonical(function.__code__), "defaults": canonical(function.__defaults__),
                "keyword_defaults": canonical(function.__kwdefaults__)}

    definitions = {}
    for name, value in list(globals().items()):
        if getattr(value, "__module__", None) != __name__:
            continue
        if isinstance(value, FunctionType):
            definitions[name] = function_signature(value)
        elif isinstance(value, type):
            definitions[name] = {method: function_signature(body) for method, body in vars(value).items()
                                 if isinstance(body, FunctionType)}
    return hashlib.sha256(json.dumps(definitions, sort_keys=True).encode("utf-8")).hexdigest()


def run_identity(args):
    # Repetitions may be extended. Live notebook edits must not mix old and new estimates.
    identity = {k: v for k, v in vars(args).items()
                if k not in ("output_dir", "restart", "fail_fast", "threads", "repetitions")}
    identity["pipeline_version"] = PIPELINE_VERSION
    identity["notebook_code"] = _implementation_digest()
    identity["sources"] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in
                           (SCRIPT_DIR / "simulate_homoscedastic.R", ROOT / "dqAux.py")}
    identity["constants"] = {"theta": THETA.tolist(), "methods": list(METHODS), "cases": list(CASES),
                             "taus": list(TAUS), "sample_sizes": list(SAMPLE_SIZES), "z975": float(Z975)}
    identity["versions"] = {"python": list(sys.version_info[:3]), "numpy": np.__version__,
                            "pandas": pd.__version__, "torch": torch.__version__}
    return identity


def row_key(row):
    return int(row["case"]), int(row["n"]), int(row["rep"]), float(row["tau"]), row["method"]


def atomic_csv(frame, path):
    temporary = path.with_suffix(".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def run(args):
    validate_args(args)
    find_rscript()
    output = args.output_dir.resolve()
    output.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(args.threads)
    torch.use_deterministic_algorithms(True)
    config = dict(mode=args.hyperparameter_mode, depth=args.depth, width=args.width, epochs=args.epochs,
                  batch_size=args.batch_size, patience=args.patience, lr=args.learning_rate)
    raw_path, config_path = output / "raw_results.csv", output / "run_config.json"
    identity = run_identity(args)
    expected = set(itertools.product(args.cases, args.sample_sizes, range(1, args.repetitions + 1),
                                     args.taus, METHODS))
    rows = []
    if not args.restart and (raw_path.exists() or (output / "data_for_r").exists()):
        if not config_path.exists() or json.loads(config_path.read_text()).get("identity") != identity:
            raise ValueError("Output settings or source code differ. Choose a fresh output directory or --restart.")
        if raw_path.exists():
            previous = pd.read_csv(raw_path, float_precision="round_trip")
            rows = previous.to_dict("records")
            keys = [row_key(row) for row in rows]
            if len(set(keys)) != len(keys) or not set(keys).issubset(expected):
                raise ValueError("Existing results have duplicate or unexpected setting keys")
            numeric = ["theta1", "theta2", "se_theta1", "se_theta2", "rmse_m", "test_check_loss"]
            if not np.isfinite(previous[numeric].to_numpy()).all():
                raise ValueError("Existing results contain invalid estimates")
        print(f"Resuming with {len(rows)} existing result rows.", flush=True)
    if args.restart:
        # Remove only this run's known result products before replacing them.
        for name in ("raw_results.csv", "simulation_summary_long.csv", "table1_bias_sd.csv",
                     "table1_bias.csv", "table1_sd.csv", "table2_coverage.csv", "table3_rmse.csv"):
            (output / name).unlink(missing_ok=True)
    metadata = vars(args).copy()
    metadata["output_dir"] = str(output)
    metadata.update(date=PIPELINE_VERSION, status="running", identity=identity,
                    requested_profile=config, expected_result_rows=len(expected),
                    depth_convention="dqAux nodes[0]; hidden layers = depth + 1",
                    inference_sample="80% estimation sample",
                    density="R stats::density defaults, old.coords=TRUE; linear interpolation at zero",
                    paper_design=dict(repetitions=200, sample_sizes=[1000, 2000], test_size=5000,
                                      taus=list(TAUS), train_validation_ratio="80:20"))
    failures = []
    start = time.perf_counter()
    completed = {row_key(row) for row in rows}

    def checkpoint(status):
        if rows:
            atomic_csv(pd.DataFrame(rows), raw_path)
        (output / "failures.json").write_text(json.dumps(failures, indent=2), encoding="utf-8")
        metadata.update(status=status, elapsed_wall_seconds=time.perf_counter() - start,
                        result_rows=len(rows), failures=len(failures), missing_result_rows=len(expected-completed))
        config_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

    checkpoint("running")
    for case, n, rep in itertools.product(args.cases, args.sample_sizes, range(1, args.repetitions + 1)):
        setting_seed = args.seed + case * 10_000_000 + n * 1_000 + rep
        rng = np.random.default_rng(setting_seed)
        x, z, y, _ = generate_dataset(n, case, rng)
        order = rng.permutation(n)
        tr, va = order[:int(0.8*n)], order[int(0.8*n):]
        xt, zt, yt, mt = generate_dataset(args.test_size, case, rng)
        if args.export_r_data:
            data_dir = output / "data_for_r"
            data_dir.mkdir(exist_ok=True)
            stem = f"case_{case}_n_{n}_rep_{rep:04d}"
            train_path, test_path = data_dir / f"{stem}_train.csv.gz", data_dir / f"{stem}_test.csv.gz"
            if args.restart or not train_path.exists():
                export_dataset(train_path, x[tr], z[tr], y[tr], nonlinear_truth(z[tr], case))
            if args.restart or not test_path.exists():
                export_dataset(test_path, xt, zt, yt, mt)
        for tau in args.taus:
            true_m_tau = mt + t.ppf(tau, df=3)
            for method in METHODS:
                key = case, n, rep, float(tau), method
                if key in completed:
                    continue
                try:
                    if method == "LQR":
                        fit = fit_lqr(x[tr], z[tr], y[tr], xt, zt, tau)
                    else:
                        fit = fit_dplqr(x[tr], z[tr], y[tr], x[va], z[va], y[va], xt, zt, tau,
                                         config, setting_seed + int(round(tau*1_000_000)))
                    theta, se, prediction, m_hat = fit[:4]
                    if not all(np.isfinite(a).all() for a in (theta, se, prediction, m_hat)) or np.any(se <= 0):
                        raise FloatingPointError("Invalid coefficient, standard error or prediction")
                    lower, upper = theta-Z975*se, theta+Z975*se
                    rows.append(dict(case=case, n=n, rep=rep, tau=tau, method=method,
                                     theta1=theta[0], theta2=theta[1], se_theta1=se[0], se_theta2=se[1],
                                     lower_theta1=lower[0], upper_theta1=upper[0],
                                     lower_theta2=lower[1], upper_theta2=upper[1],
                                     covered_theta1=lower[0] <= THETA[0] <= upper[0],
                                     covered_theta2=lower[1] <= THETA[1] <= upper[1],
                                     rmse_m=np.mean((m_hat-true_m_tau)**2)/np.mean(true_m_tau**2),
                                     test_check_loss=checkErrorMean(prediction.reshape(-1, 1), yt.reshape(-1, 1), tau=tau),
                                     selected_hyperparameters=json.dumps(fit[4]) if method == "DPLQR" else "",
                                     epochs_run=fit[5] if method == "DPLQR" else np.nan,
                                     density_zero=fit[6] if method == "DPLQR" else np.nan,
                                     projection_epochs=json.dumps(fit[7]) if method == "DPLQR" else ""))
                    completed.add(key)
                except Exception as exc:
                    failures.append(dict(case=case, n=n, rep=rep, tau=tau, method=method, error=repr(exc)))
                    checkpoint("failed" if args.fail_fast else "running")
                    if args.fail_fast:
                        raise
                checkpoint("running")
        print(f"case={case} n={n} rep={rep}/{args.repetitions}; rows={len(rows)}; "
              f"elapsed={(time.perf_counter()-start)/60:.1f} min", flush=True)
    checkpoint("complete" if completed == expected else "incomplete")
    if not rows:
        raise RuntimeError("No simulation fit completed successfully; inspect failures.json")
    summary = summarize(pd.DataFrame(rows), output)
    print(summary.to_string(index=False), flush=True)
    if completed != expected:
        raise RuntimeError("The requested experiment is incomplete. Inspect failures.json and resume before comparison.")
    return summary

## 8. Run Python: LQR and DPLQR
First use **Run All Above** on the following code cell to initialize Sections
1-7. This cell then runs the selected experiment and writes all Python result
CSVs. R `density()` is called internally for DPLQR standard errors; running the
PLAQR comparison in Section 9 is optional.

In [9]:
# Definitions may be read in any order, but execution requires the setup above.
_required_names = ('args', 'np', 'pd', 'torch', 'nn', 'tt', 'smf', 'StandardScaler', 'norm', 't', 'Path', 'CodeType', 'FunctionType', 'SimpleNamespace', 'display', 'Model', 'checkLoss', 'checkErrorMean', 'covNet', 'dqNetSparse', 'ROOT', 'SCRIPT_DIR', 'THETA', 'CASES', 'TAUS', 'SAMPLE_SIZES', 'METHODS', 'Z975', 'PIPELINE_VERSION', 'os', 'sys', 'shutil', 'subprocess', 'hashlib', 'itertools', 'json', 'math', 'random', 'time', 'find_rscript', 'residual_density_zero', 'seed_all', 'nonlinear_truth', 'generate_covariates', 'generate_dataset', 'export_dataset', 'as_numpy', 'fit_lqr', 'EarlyStopInMemory', 'clip_neural_weights', 'tensor_pair', 'train_dplqr_once', 'select_dplqr', 'fit_projection', 'dplqr_standard_errors', 'fit_dplqr', 'summarize', 'validate_args', '_implementation_digest', 'run_identity', 'row_key', 'atomic_csv', 'run')
_missing_names = [name for name in _required_names if name not in globals()]
if _missing_names:
    raise RuntimeError(
        "The notebook setup is incomplete. Use Run All Above on this cell "
        "(Sections 1-7), then run this cell. Missing: " + ", ".join(_missing_names))
python_summary = run(args)
display(python_summary)

C:\Users\Tiansui Tu\Documents\GitHub\dplqr\.venv\Lib\site-packages\torchtuples\callbacks.py:607: UserWarning: This overload of add is deprecated:
	add(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add(Tensor other, *, Number alpha = 1) (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\python_arg_parser.cpp:1841.)
  p.data = p.data.add(-weight_decay * eta, p.data)


case=1 n=1000 rep=1/2; rows=2; elapsed=0.0 min


case=1 n=1000 rep=2/2; rows=4; elapsed=0.1 min


case=2 n=1000 rep=1/2; rows=6; elapsed=0.1 min


case=2 n=1000 rep=2/2; rows=8; elapsed=0.2 min


case=3 n=1000 rep=1/2; rows=10; elapsed=0.2 min


case=3 n=1000 rep=2/2; rows=12; elapsed=0.3 min


 case    n  tau method      bias       sd  coverage     rmse  successful_reps
    1 1000  0.5  DPLQR -0.337762 0.183208       0.5 0.024519                2
    1 1000  0.5    LQR -0.028634 0.022684       1.0 0.000839                2
    2 1000  0.5  DPLQR -0.338976 0.113363       0.0 0.013805                2
    2 1000  0.5    LQR  0.009115 0.078610       1.0 0.045684                2
    3 1000  0.5  DPLQR  0.064533 0.161114       0.5 0.080518                2
    3 1000  0.5    LQR  0.198000 0.049311       1.0 0.050192                2


,case,n,tau,method,bias,sd,coverage,rmse,successful_reps
0,1,1000,0.5,DPLQR,-0.337762,0.183208,0.5,0.024519,2
1,1,1000,0.5,LQR,-0.028634,0.022684,1.0,0.000839,2
2,2,1000,0.5,DPLQR,-0.338976,0.113363,0.0,0.013805,2
3,2,1000,0.5,LQR,0.009115,0.078610,1.0,0.045684,2
4,3,1000,0.5,DPLQR,0.064533,0.161114,0.5,0.080518,2
5,3,1000,0.5,LQR,0.198000,0.049311,1.0,0.050192,2


## 9. Run R: PLAQR and combined Tables 1-3
This calls the independent R script and passes the exact same settings and output folder. It explicitly uses 95% intervals and handles both forms of the package summary.

In [10]:
if any(name not in globals() for name in ("args", "find_rscript", "subprocess", "SCRIPT_DIR")):
    raise RuntimeError("Run Sections 1-8 before starting the optional R PLAQR stage.")
if not (args.output_dir / "raw_results.csv").is_file():
    raise RuntimeError("No Python results found. Complete Section 8 before running PLAQR.")

r_command = [
    find_rscript(), "--vanilla", str(SCRIPT_DIR / "simulate_homoscedastic.R"),
    f"--repetitions={args.repetitions}",
    "--sample-sizes=" + ",".join(map(str, args.sample_sizes)),
    f"--test-size={args.test_size}",
    "--cases=" + ",".join(map(str, args.cases)),
    "--taus=" + ",".join(map(str, args.taus)),
    f"--seed={args.seed}",
    f"--output-dir={args.output_dir.resolve()}",
    f"--data-dir={args.output_dir.resolve() / 'data_for_r'}",
    f"--python-results={args.output_dir.resolve() / 'raw_results.csv'}",
]
with subprocess.Popen(r_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, errors="replace") as process:
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, r_command)

During startup - Warning messages:
1: Setting LC_COLLATE=C.UTF-8 failed 
2: Setting LC_CTYPE=C.UTF-8 failed 
3: Setting LC_MONETARY=C.UTF-8 failed 
4: Setting LC_TIME=C.UTF-8 failed 


case=1 n=1000 rep=1/2 PLAQR fits=1 elapsed=0.0 min


case=1 n=1000 rep=2/2 PLAQR fits=2 elapsed=0.0 min


case=2 n=1000 rep=1/2 PLAQR fits=3 elapsed=0.0 min


case=2 n=1000 rep=2/2 PLAQR fits=4 elapsed=0.0 min


case=3 n=1000 rep=1/2 PLAQR fits=5 elapsed=0.0 min


case=3 n=1000 rep=2/2 PLAQR fits=6 elapsed=0.0 min
    case    n tau method         bias         sd coverage         rmse
1      1 1000 0.5    LQR -0.028633710 0.02268426      1.0 0.0008391251
x1     1 1000 0.5  PLAQR -0.004332858 0.04132899      1.0 0.0025004480
2      1 1000 0.5  DPLQR -0.337762117 0.18320822      0.5 0.0245185016
5      2 1000 0.5    LQR  0.009115150 0.07860978      1.0 0.0456844112
x12    2 1000 0.5  PLAQR  0.006367098 0.06771705      1.0 0.0044142914
6      2 1000 0.5  DPLQR -0.338976383 0.11336339      0.0 0.0138049456
9      3 1000 0.5    LQR  0.198000348 0.04931147      1.0 0.0501921195
x14    3 1000 0.5  PLAQR  0.183580971 0.11255291      1.0 0.0538095327
10     3 1000 0.5  DPLQR  0.064532518 0.16111425      0.5 0.0805177982
    successful_reps          bias_sd
1                 2 -0.0286 (0.0227)
x1                2 -0.0043 (0.0413)
2                 2 -0.3378 (0.1832)
5                 2  0.0091 (0.0786)
x12               2  0.0064 (0.0677)
6                

## 10. Display reproduced and published tables
Published CSVs are transcription targets. They are displayed separately and are never used to generate or replace simulated estimates. A short run does not provide a reliable Monte Carlo comparison.

In [11]:
if any(name not in globals() for name in ("args", "pd", "display", "SCRIPT_DIR")):
    raise RuntimeError("Run the setup cells and Section 8 before displaying result tables.")
_expected_tables = ["table1_bias_sd.csv", "table2_coverage.csv", "table3_rmse.csv"]
if any(not (args.output_dir / name).is_file() for name in _expected_tables):
    raise RuntimeError("Result tables are not available. Complete Section 8 first.")

from IPython.display import Markdown
for number, filename in [(1, "table1_bias_sd.csv"), (2, "table2_coverage.csv"), (3, "table3_rmse.csv")]:
    display(Markdown(f"### Table {number}: reproduced ({args.repetitions} repetitions)"))
    display(pd.read_csv(args.output_dir / filename))
    display(Markdown(f"### Table {number}: published in Zhong and Wang (2024)"))
    display(pd.read_csv(SCRIPT_DIR / f"published_table{number}.csv"))

### Table 1: reproduced (2 repetitions)

,case,n,tau_0.50_LQR,tau_0.50_PLAQR,tau_0.50_DPLQR
0,1,1000,-0.0286 (0.0227),-0.0043 (0.0413),-0.3378 (0.1832)
1,2,1000,0.0091 (0.0786),0.0064 (0.0677),-0.3390 (0.1134)
2,3,1000,0.1980 (0.0493),0.1836 (0.1126),0.0645 (0.1611)


### Table 1: published in Zhong and Wang (2024)

,case,n,tau_0.25_LQR,tau_0.25_PLAQR,tau_0.25_DPLQR,tau_0.50_LQR,tau_0.50_PLAQR,tau_0.50_DPLQR,tau_0.75_LQR,tau_0.75_PLAQR,tau_0.75_DPLQR
0,1,1000,0.0896 (0.1402),0.0944 (0.1428),0.1235 (0.1446),0.0219 (0.1116),0.0413 (0.1199),0.0636 (0.1235),-0.0721 (0.1389),0.0836 (0.1411),0.1203 (0.1439)
1,1,2000,0.0742 (0.0970),0.0821 (0.0998),0.1078 (0.1074),-0.0148 (0.0796),0.0304 (0.0812),0.0483 (0.0925),-0.0679 (0.0988),-0.0795 (0.1003),0.0983 (0.1118)
2,2,1000,0.0764 (0.1384),-0.0297 (0.1253),0.0360 (0.1301),-0.0611 (0.1158),-0.0124 (0.1008),0.0338 (0.1116),-0.0757 (0.1493),-0.0210 (0.1364),-0.0307 (0.1417)
3,2,2000,0.0710 (0.1086),-0.0145 (0.0970),0.0229 (0.0976),0.0506 (0.0794),0.0099 (0.0782),0.0158 (0.0785),-0.0727 (0.0943),0.0112 (0.1060),-0.0266 (0.0984)
4,3,1000,0.1244 (0.1787),-0.1060 (0.1647),0.0762 (0.1394),0.1068 (0.1444),-0.0902 (0.1337),0.0403 (0.0998),-0.1322 (0.1718),-0.1095 (0.1691),0.0604 (0.1330)
5,3,2000,0.1170 (0.1231),-0.0935 (0.1146),0.0556 (0.0951),-0.0951 (0.1046),-0.0630 (0.0978),0.0297 (0.0848),-0.1212 (0.1165),-0.0942 (0.1146),-0.0445 (0.0941)


### Table 2: reproduced (2 repetitions)

...

,case,n,tau_0.25_LQR,tau_0.25_PLAQR,tau_0.25_DPLQR,tau_0.50_LQR,tau_0.50_PLAQR,tau_0.50_DPLQR,tau_0.75_LQR,tau_0.75_PLAQR,tau_0.75_DPLQR
0,1,1000,0.910,0.900,0.895,0.975,0.915,0.905,0.900,0.910,0.905
1,1,2000,0.925,0.920,0.915,0.945,0.940,0.935,0.930,0.920,0.925
2,2,1000,0.855,0.910,0.895,0.960,0.955,0.945,0.865,0.915,0.910
3,2,2000,0.885,0.925,0.915,0.960,0.945,0.955,0.895,0.935,0.930
4,3,1000,0.865,0.885,0.910,0.885,0.875,0.930,0.870,0.895,0.925
5,3,2000,0.900,0.925,0.935,0.915,0.920,0.955,0.905,0.920,0.945


### Table 3: reproduced (2 repetitions)

,case,n,tau_0.50_LQR,tau_0.50_PLAQR,tau_0.50_DPLQR
0,1,1000,0.000839,0.002500,0.024519
1,2,1000,0.045684,0.004414,0.013805
2,3,1000,0.050192,0.053810,0.080518


### Table 3: published in Zhong and Wang (2024)

,case,n,tau_0.25_LQR,tau_0.25_PLAQR,tau_0.25_DPLQR,tau_0.50_LQR,tau_0.50_PLAQR,tau_0.50_DPLQR,tau_0.75_LQR,tau_0.75_PLAQR,tau_0.75_DPLQR
0,1,1000,0.0071,0.0080,0.0086,0.0064,0.0070,0.0081,0.0074,0.0083,0.0089
1,1,2000,0.0036,0.0043,0.0047,0.0034,0.0038,0.0040,0.0037,0.0044,0.0049
2,2,1000,0.0097,0.0072,0.0074,0.0062,0.0053,0.0059,0.0093,0.0074,0.0080
3,2,2000,0.0059,0.0040,0.0044,0.0034,0.0027,0.0030,0.0051,0.0039,0.0045
4,3,1000,0.0997,0.0872,0.0390,0.0679,0.0539,0.0209,0.0976,0.0820,0.0342
5,3,2000,0.0904,0.0812,0.0298,0.0633,0.0508,0.0159,0.0883,0.0774,0.0275
